In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import os

from itertools import combinations
from matplotlib.colors import to_rgb

# imports from "common" folder
from common.data_processing import process_data

In [ ]:
# Compute the relative Error matrix for each site
RF  = pd.read_csv("../data/SHAP_RF_df.csv", index_col=0)
BRT = pd.read_csv("../data/SHAP_BRT_df.csv", index_col=0)
MLP = pd.read_csv("../data/SHAP_MLP_df.csv", index_col=0)
GAM = pd.read_csv("../data/SHAP_GAM_df.csv", index_col=0)


# remove year, longititude and latitude
cols_to_drop = ["year", "longitude", "latitude"]
RF = RF.drop(columns=cols_to_drop)
BRT = BRT.drop(columns=cols_to_drop)
MLP = MLP.drop(columns=cols_to_drop)
GAM = GAM.drop(columns=cols_to_drop)

# Organize into dictionary
models = ['GAM','RF', 'BRT', 'MLP']
shap_data = {
    'GAM': GAM.values,
    'RF': RF.values,
    'BRT': BRT.values,
    'MLP': MLP.values,
    
}

# Compute relative error matrices per site
site_relative_error_matrices = {}

for site_idx in range(2778):  # 0-based index
    matrix = pd.DataFrame(index=models, columns=models, dtype=float)

    for ref_model in models:
        for cmp_model in models:
            if ref_model == cmp_model:
                matrix.loc[ref_model, cmp_model] = np.nan  
            else:
                S_ref = shap_data[ref_model][site_idx]
                S_cmp = shap_data[cmp_model][site_idx]
                norm_ref = np.linalg.norm(S_ref)
                rel_error = np.linalg.norm(S_ref - S_cmp) / norm_ref if norm_ref != 0 else np.nan
                matrix.loc[ref_model, cmp_model] = round(rel_error, 3)


    matrix.index.name = f"Compared Model"
    site_relative_error_matrices[site_idx + 1] = matrix

for site_id in range(1, 2779): 
    print(f"\nRelative Error Matrix for Site {site_id}:")
    print(site_relative_error_matrices[site_id])


In [ ]:
# Compute the averageRelative error per site - accroding to the reference model
avg_error_per_site_ref = []

for site_id, matrix in site_relative_error_matrices.items():
    for ref_model in models:
        row = matrix.loc[ref_model]
        valid_errors = row.dropna()
        avg_error = valid_errors.mean()
       
        avg_error_per_site_ref.append({
            "Site": site_id,
            "Reference_Model": ref_model,
            "Average_Relative_Error": round(avg_error, 3)
        })

avg_error_ref_df = pd.DataFrame(avg_error_per_site_ref)

print(avg_error_ref_df)

In [ ]:
avg_error_ref_df['Rank'] = avg_error_ref_df.groupby('Reference_Model')['Average_Relative_Error'] \
                                           .rank(method='min', ascending=False)

avg_error_ref_ranked_df = avg_error_ref_df.sort_values(['Reference_Model', 'Rank'])

avg_error_ref_ranked_df


#### Boxplot of projected SHAP discrepancy

In [ ]:
# Plot the boxplot of EDM (SHAP discrepancy)
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    "legend.fontsize": 12,
    "legend.title_fontsize": 14,
    "legend.frameon": False,
})


model_order = ['GAM','RF', 'BRT', 'MLP']
ref_palette = {
    'GAM': sns.color_palette("Purples", 4)[1],
    'RF':  sns.color_palette("Blues", 4)[1],
    'BRT': sns.color_palette("Greens", 4)[1],
    'MLP': sns.color_palette("Reds", 4)[1]
}

records = []
for model in model_order:
    sub_df = avg_error_ref_ranked_df[avg_error_ref_ranked_df['Reference_Model'] == model].copy()
    sd = sub_df['Average_Relative_Error'].std(ddof=1)
    sub_df['Mean_over_SDofMean'] = sub_df['Average_Relative_Error'] / sd if sd != 0 else np.nan
    records.append(sub_df)

df = pd.concat(records).dropna(subset=['Mean_over_SDofMean'])


plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df,
    x='Reference_Model',
    y='Mean_over_SDofMean',
    hue='Reference_Model',
    order=model_order,
    palette=ref_palette,
    showfliers=False
)


jitter_strength = 0.15  
np.random.seed(42)  

for i, model in enumerate(model_order):
    model_df = df[df['Reference_Model'] == model]
    q3 = model_df['Mean_over_SDofMean'].quantile(0.75)
    q1 = model_df['Mean_over_SDofMean'].quantile(0.25)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outliers = model_df[model_df['Mean_over_SDofMean'] > upper_bound]

   
    x_vals = i + np.random.uniform(-jitter_strength, jitter_strength, size=len(model_df))
    y_vals = model_df['Mean_over_SDofMean'].values
    sites = model_df['Site'].values

   
    # dot_color = ref_palette[model]
    base_color = to_rgb(ref_palette[model])
    dot_color = tuple([0.8 * c for c in base_color])  

    plt.scatter(x_vals, y_vals, color = dot_color, alpha=0.6, s=30, zorder=2)


    # for xi, yi, site in zip(x_vals, y_vals, sites):
    #     if yi > upper_bound:
    #         plt.text(
    #             xi + 0.04,
    #             yi,
    #             str(site),
    #             fontsize=9,
    #             color='red',
    #             rotation=45,
    #             ha='left',
    #             va='center',
    #             zorder=3
    #         )


plt.xlabel("Reference Model", fontsize=16,labelpad=10)
# plt.ylabel("Mean / SD(Mean)", fontsize=14)
# plt.ylabel(r"$\mu(i)\,/ \,\sigma_{\mu}$", fontsize=16, labelpad=10)
# plt.title("Training", fontsize=16)
# plt.ylabel(r"$\mu_{j_{1}} (i)/\sigma_{\mu_{j_{1}}}$", fontsize=16, labelpad=10)
plt.ylabel("Projected SHAP discrepancy", fontsize=16, labelpad=10)
plt.tick_params(axis = 'x', labelsize =14)
plt.tick_params(axis = 'y', labelsize =14)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()



In [ ]:
# dataset with Mean/SD 
records = []
for model in model_order:
    sub_df = avg_error_ref_ranked_df[avg_error_ref_ranked_df['Reference_Model'] == model].copy()
    sd = sub_df['Average_Relative_Error'].std(ddof=1)
    sub_df['Mean_over_SDofMean'] = sub_df['Average_Relative_Error'] / sd if sd != 0 else np.nan
    records.append(sub_df)

df = pd.concat(records).dropna(subset=['Mean_over_SDofMean'])

# Save SHAP discrepancy 
df.to_csv("../data/projected_SHAP_discrepancy_training_table.csv", index=False)

# Collect outliers per model
outlier_dict = {}

for model in model_order:
    model_df = df[df['Reference_Model'] == model]
    q3 = model_df['Mean_over_SDofMean'].quantile(0.75)
    q1 = model_df['Mean_over_SDofMean'].quantile(0.25)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outliers = model_df.loc[model_df['Mean_over_SDofMean'] > upper_bound, 'Site'].tolist()
    outlier_dict[model] = outliers

# Save the indices of outliers for all models
outliers_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in outlier_dict.items()]))
print(outliers_df)
outliers_df.to_csv("../data/train_outliers_table_projected.csv", index=False)


In [ ]:
datasets = {
    model: df[df["Reference_Model"] == model].sort_values(by="Site")
    for model in df["Reference_Model"].unique()
}

df_gam = datasets["GAM"]
df_rf  = datasets["RF"]
df_brt = datasets["BRT"]
df_mlp = datasets["MLP"]

# Save SHAP discrepancy by reference model
for model, dset in datasets.items():
    dset.to_csv(f"../data/{model}_ref_model_projected_SHAP_discrepancy.csv", index=False)
